# TabPFN comparator runs (one per use case)

Runs [TabPFN](https://github.com/PriorLabs/TabPFN), Prior Labs' pretrained
tabular foundation model, against the same train/test splits and metadata
enrichment as the ritme and AutoML arms, via `submit_tabpfn(...)` from
`src/launch_tabpfn.py`. TabPFN predicts in context from the training set and
has no hyperparameter search, so there is no time budget: a run takes minutes
on CPU.

The feature tables (11k-36k features) are far wider than TabPFN's pretraining
budget of 500 features, so each use case runs twice:

- **all features** - the full table goes in. TabPFN's own feature subsampling
  hands every ensemble member the metadata columns plus a balanced slice of the
  microbial features, with enough members for every feature to be seen at
  least once.
- **top-500** - the design matrix is capped at 500 columns: the metadata plus
  the most abundant microbial features of the training split. TabPFN then runs
  within its pretraining limits with default settings.

The data splits (`data_splits_*/train_val.pkl`, `test.pkl`) are produced by
`submit_model(...)` from `src/launch_models.py`; run those once first.

Run this notebook from the `tabpfn` environment, created once from the repo
root:

```shell
mamba env create -f envs/tabpfn.yml -p $CONDA_ENVS/tabpfn
mamba run -n tabpfn pip install -e .
mamba run -n tabpfn python -m ipykernel install --user --name tabpfn
```

TabPFN v2 (Hollmann et al., Nature 2025) is used because its weights download
without an account. Later versions are gated behind the Prior Labs license:
accept it at https://ux.priorlabs.ai, export `TABPFN_TOKEN` and `HF_TOKEN`,
and set `MODEL_VERSION` below.

## Setup

In [1]:
from src.compare_test_metrics import (
    collect,
    format_table,
    plot_comparison,
    tabpfn_settings,
)
from src.launch_tabpfn import ensure_models, metrics_path, submit_tabpfn

## Configuration

In [2]:
# Where the TabPFN outputs and SLURM logs land, next to the other comparator arms.
LOGS_DIR = "comparators"

# TabPFN release and the width of its pretraining budget (features).
MODEL_VERSION = "v2"
MAX_FEATURES = 500

# Per-job SLURM resources. The account and node type come from .cluster.json
# (gitignored); see src/cluster_config.py.
CPUS = 8
MEM_PER_CPU_MB = 8192

common = dict(
    mode="slurm",
    logs_dir=LOGS_DIR,
    cpus=CPUS,
    mem_per_cpu_mb=MEM_PER_CPU_MB,
    model_version=MODEL_VERSION,
)

## Model weights

Compute nodes have no internet access. Download the checkpoints once from a
node that has; the jobs read them from `external/tabpfn_models/` (gitignored).

In [3]:
ensure_models(model_version=MODEL_VERSION)

TabPFN v2 checkpoints in external/tabpfn_models


## Submit the two TabPFN arms per use case

Arms whose metrics file already exists are not resubmitted, so the notebook can
be re-run end to end once the jobs have finished.

In [4]:
for usecase in ("u1", "u2", "u3"):
    for max_features in (None, MAX_FEATURES):
        done = metrics_path(usecase, max_features, LOGS_DIR)
        if done.exists():
            print(f"{done.name} exists, not resubmitted")
            continue
        submit_tabpfn(usecase, max_features=max_features, **common)

u1_tabpfn_metrics.csv exists, not resubmitted
u1_tabpfn_top500_metrics.csv exists, not resubmitted
u2_tabpfn_metrics.csv exists, not resubmitted
u2_tabpfn_top500_metrics.csv exists, not resubmitted
u3_tabpfn_metrics.csv exists, not resubmitted
u3_tabpfn_top500_metrics.csv exists, not resubmitted


## Results

Once the jobs have finished, the held-out test metrics of both TabPFN arms are
compared with ritme's model of record per use case (the manuscript's
selection: ritme's 1-SE pick per model type, ranked by mean validation score,
as in `evaluate_all_trials.ipynb`) and with the other comparator arms. Brackets
are 95% percentile-bootstrap intervals over the test split (1000 resamples),
computed the same way for every arm. F1 is the positive-class F1 at the 0.5
threshold; macro-F1 averages both classes.

In [5]:
tabpfn_settings()

n_features  n_metadata_features  \
usecase arm                                                          
u1      TabPFN (all features)           18156                    6   
        TabPFN (top-500 features)         500                    6   
u2      TabPFN (all features)           35659                    8   
        TabPFN (top-500 features)         500                    8   
u3      TabPFN (all features)           11285                    3   
        TabPFN (top-500 features)         500                    3   

                                   n_estimators    feature_subsampling  \
usecase arm                                                              
u1      TabPFN (all features)                37  constant_and_balanced   
        TabPFN (top-500 features)             8                    NaN   
u2      TabPFN (all features)                73  constant_and_balanced   
        TabPFN (top-500 features)             8                    NaN   
u3      TabPFN (all features)                23  constant_and_balanced   
        TabPFN (top-500 features)             8                    NaN   

                                   fit_seconds  predict_seconds  n_threads  \
usecase arm                                                                  
u1      TabPFN (all features)             33.2           1286.5          8   
        TabPFN (top-500 features)          6.2            322.9          8   
u2      TabPFN (all features)             52.8            742.6          8   
        TabPFN (top-500 features)          3.5            105.7          8   
u3      TabPFN (all features)             12.4            730.8          8   
        TabPFN (top-500 features)          0.9            252.8          8   

                                   slurm_job_id  
usecase arm                                      
u1      TabPFN (all features)          20316775  
        TabPFN (top-500 features)      20316790  
u2      TabPFN (all features)          20316795  
        TabPFN (top-500 features)      20316798  
u3      TabPFN (all features)          20316800  
        TabPFN (top-500 features)      20316802

In [6]:
tidy = collect()
ritme_vs_tabpfn = tidy[tidy["arm"].str.startswith(("ritme", "tabpfn"))]
format_table(ritme_vs_tabpfn)

metric                                                   RMSE (↓)  \
usecase arm                       model                             
u1      ritme                     xgb        2.892 [2.374, 3.396]   
        TabPFN (all features)     TabPFN-v2  3.290 [2.879, 3.703]   
        TabPFN (top-500 features) TabPFN-v2  2.950 [2.340, 3.494]   
u2      ritme                     linreg     1.435 [0.931, 1.986]   
        TabPFN (all features)     TabPFN-v2  1.462 [0.881, 2.081]   
        TabPFN (top-500 features) TabPFN-v2  1.522 [0.918, 2.113]   
u3      ritme                     logreg                            
        TabPFN (all features)     TabPFN-v2                         
        TabPFN (top-500 features) TabPFN-v2                         

metric                                                     R² (↑)  \
usecase arm                       model                             
u1      ritme                     xgb        0.773 [0.645, 0.857]   
        TabPFN (all features)     TabPFN-v2  0.707 [0.583, 0.785]   
        TabPFN (top-500 features) TabPFN-v2  0.764 [0.610, 0.861]   
u2      ritme                     linreg     0.957 [0.892, 0.983]   
        TabPFN (all features)     TabPFN-v2  0.955 [0.879, 0.985]   
        TabPFN (top-500 features) TabPFN-v2  0.952 [0.874, 0.985]   
u3      ritme                     logreg                            
        TabPFN (all features)     TabPFN-v2                         
        TabPFN (top-500 features) TabPFN-v2                         

metric                                                  AUROC (↑)  \
usecase arm                       model                             
u1      ritme                     xgb                               
        TabPFN (all features)     TabPFN-v2                         
        TabPFN (top-500 features) TabPFN-v2                         
u2      ritme                     linreg                            
        TabPFN (all features)     TabPFN-v2                         
        TabPFN (top-500 features) TabPFN-v2                         
u3      ritme                     logreg     0.770 [0.667, 0.856]   
        TabPFN (all features)     TabPFN-v2  0.816 [0.728, 0.890]   
        TabPFN (top-500 features) TabPFN-v2  0.820 [0.737, 0.897]   

metric                                                     F1 (↑) macro-F1 (↑)  
usecase arm                       model                                         
u1      ritme                     xgb                                           
        TabPFN (all features)     TabPFN-v2                                     
        TabPFN (top-500 features) TabPFN-v2                                     
u2      ritme                     linreg                                        
        TabPFN (all features)     TabPFN-v2                                     
        TabPFN (top-500 features) TabPFN-v2                                     
u3      ritme                     logreg     0.492 [0.338, 0.632]        0.620  
        TabPFN (all features)     TabPFN-v2  0.674 [0.558, 0.779]        0.702  
        TabPFN (top-500 features) TabPFN-v2  0.613 [0.466, 0.736]        0.687

All arms, for context:

In [7]:
format_table(tidy)

metric                                                                                            RMSE (↓)  \
usecase arm                       model                                                                      
u1      ritme                     xgb                                                 2.892 [2.374, 3.396]   
        TabPFN (all features)     TabPFN-v2                                           3.290 [2.879, 3.703]   
        TabPFN (top-500 features) TabPFN-v2                                           2.950 [2.340, 3.494]   
        auto-sklearn                                                                                 2.960   
        TPOT                      xgboost.XGBRegressor                                               2.826   
        original study            rf                                                  3.498 [3.043, 4.002]   
u2      ritme                     linreg                                              1.435 [0.931, 1.986]   
        TabPFN (all features)     TabPFN-v2                                           1.462 [0.881, 2.081]   
        TabPFN (top-500 features) TabPFN-v2                                           1.522 [0.918, 2.113]   
        auto-sklearn                                                                                 2.048   
        TPOT                      sklearn.linear_model.ElasticNetCV                                  1.906   
        original study            elasticnet                                          2.011 [1.306, 2.791]   
u3      ritme                     logreg                                                                     
        TabPFN (all features)     TabPFN-v2                                                                  
        TabPFN (top-500 features) TabPFN-v2                                                                  
        auto-sklearn                                                                                         
        TPOT                      sklearn.linear_model.LogisticRegression                                    
        mAML                      u3_train_X.csv.NonScaler_BaggingClassifier.model.z                         
        original study            rf_class                                                                   

metric                                                                                              R² (↑)  \
usecase arm                       model                                                                      
u1      ritme                     xgb                                                 0.773 [0.645, 0.857]   
        TabPFN (all features)     TabPFN-v2                                           0.707 [0.583, 0.785]   
        TabPFN (top-500 features) TabPFN-v2                                           0.764 [0.610, 0.861]   
        auto-sklearn                                                                                 0.763   
        TPOT                      xgboost.XGBRegressor                                               0.784   
        original study            rf                                                  0.669 [0.503, 0.763]   
u2      ritme                     linreg                                              0.957 [0.892, 0.983]   
        TabPFN (all features)     TabPFN-v2                                           0.955 [0.879, 0.985]   
        TabPFN (top-500 features) TabPFN-v2                                           0.952 [0.874, 0.985]   
        auto-sklearn                                                                                 0.912   
        TPOT                      sklearn.linear_model.ElasticNetCV                                  0.924   
        original study            elasticnet                                          0.915 [0.788, 0.968]   
u3      ritme                     logreg                                                                     
        TabPFN (all features)     TabPF

In [8]:
fig = plot_comparison(tidy)
fig.savefig("result_figures/tabpfn_comparison.pdf", bbox_inches="tight")
fig.savefig("result_figures/tabpfn_comparison.png", dpi=300, bbox_inches="tight")